# FitoLens: entrenar router

1. Panel derecho → **Settings**: *Accelerator* = **GPU T4 x2** o **GPU P100**, *Internet* = **On**.
2. Menú **Save Version** → **Save & Run All (Commit)**. Puedes cerrar la pestaña: corre solo.
3. Al terminar, en la pestaña **Output** descarga `salida/modelo_<nombre>.zip`.

El código se genera desde `entrenamiento/fitolens_ent/` del repositorio FitoLens: no lo edites aquí.

In [ ]:
!pip install -q timm onnx onnxruntime imagehash kagglehub
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import json, os
os.makedirs('fitolens_ent', exist_ok=True)
ARCHIVOS = json.loads("{\"__init__.py\": \"\", \"aumentos.py\": \"\\\"\\\"\\\"Aumentos que imitan fotos de campo, y la carga de im\u00e1genes id\u00e9ntica a la de la API.\\\"\\\"\\\"\\nimport io, random\\n\\nimport numpy as np\\nimport torch\\nfrom PIL import Image, ImageFilter, ImageOps\\nfrom torchvision.transforms import v2 as T\\n\\n\\ndef cargar_como_api(img, tam):\\n    \\\"\\\"\\\"Exactamente lo que hace la API: RGB, orientaci\u00f3n EXIF, redimensi\u00f3n directa (BILINEAR), 0-1, NCHW.\\\"\\\"\\\"\\n    img = ImageOps.exif_transpose(img).convert(\\\"RGB\\\").resize((tam, tam), Image.BILINEAR)\\n    return torch.from_numpy(np.asarray(img, np.float32) / 255.0).permute(2, 0, 1)\\n\\n\\nclass ReemplazoFondo:\\n    \\\"\\\"\\\"En fotos de laboratorio (fondo liso) cambia el fondo por un recorte de una foto de campo.\\n\\n    La hoja se separa por saturaci\u00f3n (umbral de Otsu) y se rellenan los huecos para no borrar lesiones.\\n    \\\"\\\"\\\"\\n\\n    def __init__(self, fondos, p=0.6):\\n        self.fondos, self.p = fondos, p\\n\\n    def __call__(self, img):\\n        if not self.fondos or random.random() > self.p:\\n            return img\\n        from scipy import ndimage\\n        a = np.asarray(img.convert(\\\"HSV\\\"))[..., 1].astype(np.float32)\\n        hist = np.histogram(a, 256, (0, 256))[0].astype(np.float64)\\n        w = np.cumsum(hist); mu = np.cumsum(hist * np.arange(256))\\n        var = (mu[-1] * w / w[-1] - mu) ** 2 / (w * (w[-1] - w) + 1e-9)\\n        mascara = a > np.argmax(var)\\n        mascara = ndimage.binary_fill_holes(ndimage.binary_closing(mascara, iterations=3))\\n        etiq, n = ndimage.label(mascara)\\n        if n == 0:\\n            return img\\n        mayor = np.argmax(ndimage.sum(mascara, etiq, range(1, n + 1))) + 1\\n        mascara = etiq == mayor\\n        if not 0.08 < mascara.mean() < 0.95:\\n            return img\\n        fondo = Image.open(random.choice(self.fondos)).convert(\\\"RGB\\\")\\n        fondo = T.RandomResizedCrop(img.size[::-1], scale=(0.2, 1.0))(fondo)\\n        alfa = Image.fromarray((mascara * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(2))\\n        return Image.composite(img, fondo, alfa)\\n\\n\\nclass JPEGAleatorio:\\n    def __init__(self, p=0.4, q=(30, 90)):\\n        self.p, self.q = p, q\\n\\n    def __call__(self, img):\\n        if random.random() > self.p:\\n            return img\\n        b = io.BytesIO()\\n        img.save(b, \\\"JPEG\\\", quality=random.randint(*self.q))\\n        return Image.open(b).convert(\\\"RGB\\\")\\n\\n\\ndef entrenamiento(tam):\\n    return T.Compose([\\n        T.RandomResizedCrop(tam, scale=(0.35, 1.0), ratio=(0.75, 1.33), antialias=True),\\n        T.RandomHorizontalFlip(), T.RandomVerticalFlip(0.2), T.RandomRotation(20),\\n        T.ColorJitter(0.4, 0.4, 0.3, 0.05),\\n        T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.25),\\n        JPEGAleatorio(),\\n        T.PILToTensor(), T.ToDtype(torch.float32, scale=True),\\n        T.RandomErasing(p=0.2, scale=(0.02, 0.12)),\\n    ])\\n\\n\\nclass Conjunto(torch.utils.data.Dataset):\\n    def __init__(self, muestras, tam, entrenar=False, fondos=None):\\n        self.m, self.tam, self.entrenar = muestras, tam, entrenar\\n        self.aug = entrenamiento(tam) if entrenar else None\\n        self.fondo = ReemplazoFondo(fondos or [])\\n\\n    def __len__(self):\\n        return len(self.m)\\n\\n    def __getitem__(self, i):\\n        m = self.m[i]\\n        img = Image.open(m[\\\"ruta\\\"]).convert(\\\"RGB\\\")\\n        if self.entrenar:\\n            if not m[\\\"campo\\\"]:\\n                img = self.fondo(img)\\n            return self.aug(img), m[\\\"y\\\"]\\n        return cargar_como_api(img, self.tam), m[\\\"y\\\"]\\n\", \"catalogo.py\": \"\\\"\\\"\\\"Cat\u00e1logo \u00fanico de FitoLens: clases, fuentes de datos y receta de cada modelo.\\n\\nPara mantener el proyecto, casi todo cambio se hace aqu\u00ed:\\n- agregar una enfermedad  -> a\u00f1adirla a CLASES[...] y una regla en reglas.py\\n- agregar un dataset      -> a\u00f1adir una entrada en FUENTES\\n- cambiar la receta       -> editar RECETA / RECETA_POR_MODELO\\n\\\"\\\"\\\"\\n\\n# ---------------------------------------------------------------- clases\\n# ids en snake_case, prefijo de cultivo y \\\"_sana\\\" para las sanas (contrato con la API y la web)\\nCLASES = {\\n    \\\"router\\\": [\\\"pimiento\\\", \\\"papa\\\", \\\"tomate\\\", \\\"cafe\\\", \\\"cacao\\\", \\\"otra_planta\\\", \\\"no_planta\\\"],\\n    \\\"hortalizas\\\": [\\n        \\\"pimiento_mancha_bacteriana\\\", \\\"pimiento_sana\\\",\\n        \\\"papa_tizon_temprano\\\", \\\"papa_tizon_tardio\\\", \\\"papa_sana\\\",\\n        \\\"tomate_mancha_bacteriana\\\", \\\"tomate_tizon_temprano\\\", \\\"tomate_tizon_tardio\\\",\\n        \\\"tomate_moho_hoja\\\", \\\"tomate_septoriosis\\\", \\\"tomate_arana_roja\\\", \\\"tomate_mancha_diana\\\",\\n        \\\"tomate_virus_rizado_amarillo\\\", \\\"tomate_virus_mosaico\\\", \\\"tomate_sana\\\",\\n    ],\\n    \\\"cafe\\\": [\\\"cafe_sana\\\", \\\"cafe_roya\\\", \\\"cafe_ojo_de_gallo\\\"],\\n    \\\"cacao\\\": [\\\"cacao_sana\\\", \\\"cacao_monilia\\\", \\\"cacao_mazorca_negra\\\"],\\n}\\n\\nETIQUETAS = {\\n    \\\"pimiento\\\": \\\"Pimiento\\\", \\\"papa\\\": \\\"Papa\\\", \\\"tomate\\\": \\\"Tomate\\\", \\\"cafe\\\": \\\"Caf\u00e9\\\", \\\"cacao\\\": \\\"Cacao\\\",\\n    \\\"otra_planta\\\": \\\"Otra planta (no soportada)\\\", \\\"no_planta\\\": \\\"No es una planta\\\",\\n    \\\"pimiento_mancha_bacteriana\\\": \\\"Pimiento: mancha bacteriana\\\", \\\"pimiento_sana\\\": \\\"Pimiento: sana\\\",\\n    \\\"papa_tizon_temprano\\\": \\\"Papa: tiz\u00f3n temprano\\\", \\\"papa_tizon_tardio\\\": \\\"Papa: tiz\u00f3n tard\u00edo (rancha)\\\",\\n    \\\"papa_sana\\\": \\\"Papa: sana\\\",\\n    \\\"tomate_mancha_bacteriana\\\": \\\"Tomate: mancha bacteriana\\\", \\\"tomate_tizon_temprano\\\": \\\"Tomate: tiz\u00f3n temprano\\\",\\n    \\\"tomate_tizon_tardio\\\": \\\"Tomate: tiz\u00f3n tard\u00edo\\\", \\\"tomate_moho_hoja\\\": \\\"Tomate: moho de la hoja\\\",\\n    \\\"tomate_septoriosis\\\": \\\"Tomate: septoriosis\\\", \\\"tomate_arana_roja\\\": \\\"Tomate: ara\u00f1a roja\\\",\\n    \\\"tomate_mancha_diana\\\": \\\"Tomate: mancha diana\\\", \\\"tomate_virus_rizado_amarillo\\\": \\\"Tomate: virus del rizado amarillo\\\",\\n    \\\"tomate_virus_mosaico\\\": \\\"Tomate: virus del mosaico\\\", \\\"tomate_sana\\\": \\\"Tomate: sana\\\",\\n    \\\"cafe_sana\\\": \\\"Caf\u00e9: sana\\\", \\\"cafe_roya\\\": \\\"Caf\u00e9: roya\\\", \\\"cafe_ojo_de_gallo\\\": \\\"Caf\u00e9: ojo de gallo\\\",\\n    \\\"cacao_sana\\\": \\\"Cacao: sana\\\", \\\"cacao_monilia\\\": \\\"Cacao: moniliasis\\\", \\\"cacao_mazorca_negra\\\": \\\"Cacao: mazorca negra\\\",\\n}\\n\\nESPECIALISTA_DE_CULTIVO = {\\\"pimiento\\\": \\\"hortalizas\\\", \\\"papa\\\": \\\"hortalizas\\\", \\\"tomate\\\": \\\"hortalizas\\\",\\n                           \\\"cafe\\\": \\\"cafe\\\", \\\"cacao\\\": \\\"cacao\\\"}\\nRECHAZO = [\\\"otra_planta\\\", \\\"no_planta\\\"]\\n\\n# ---------------------------------------------------------------- fuentes\\n# tipo: kaggle (kagglehub) | git | mendeley | url\\n# campo: True si son fotos reales de campo (solo estas entran en validaci\u00f3n y prueba)\\n# regla: funci\u00f3n de reglas.py que convierte la ruta en etiqueta\\n# requerido: si falla la descarga, el entrenamiento se detiene (False: solo avisa)\\n# Respaldo manual: si una descarga falla, sube el archivo como dataset de Kaggle\\n# llamado \\\"fitolens-<clave>\\\" y agr\u00e9galo al notebook; se busca primero en /kaggle/input.\\nFUENTES = {\\n    \\\"plantvillage\\\": dict(\\n        tipo=\\\"kaggle\\\", ref=\\\"abdallahalidev/plantvillage-dataset\\\", campo=False, regla=\\\"hortalizas\\\",\\n        incluir=\\\"/color/\\\", modelos=[\\\"hortalizas\\\", \\\"router\\\"], requerido=True,\\n        cita=\\\"Hughes y Salath\u00e9 (2015), PlantVillage. Copia en Kaggle: abdallahalidev/plantvillage-dataset\\\",\\n        licencia=\\\"ver la fuente original (PlantVillage)\\\"),\\n    \\\"plantdoc\\\": dict(\\n        tipo=\\\"git\\\", url=\\\"https://github.com/pratikkayal/PlantDoc-Dataset.git\\\", campo=True, regla=\\\"hortalizas\\\",\\n        split_por_carpeta=True, sin_enfermedad_es_sana=True, modelos=[\\\"hortalizas\\\", \\\"router\\\"], requerido=True,\\n        cita=\\\"Singh et al. (2020), PlantDoc, CODS-COMAD\\\", licencia=\\\"CC BY 4.0\\\"),\\n    \\\"papa_indonesia\\\": dict(\\n        tipo=\\\"mendeley\\\", id=\\\"ptz377bwb8\\\", version=1, campo=True, regla=\\\"papa_indonesia\\\",\\n        modelos=[\\\"hortalizas\\\", \\\"router\\\"], requerido=False,\\n        cita=\\\"Potato Leaf Disease Dataset in Uncontrolled Environment, Mendeley Data, 10.17632/ptz377bwb8.1\\\",\\n        licencia=\\\"CC BY 4.0\\\"),\\n    \\\"tomate_bangladesh\\\": dict(\\n        tipo=\\\"mendeley\\\", id=\\\"rnbsw72zb5\\\", version=1, campo=True, regla=\\\"solo_tomate\\\",\\n        modelos=[\\\"router\\\"], requerido=False,\\n        cita=\\\"Tomato Leaf Image Dataset for Disease Analysis in Real-World Environment, 10.17632/rnbsw72zb5.1\\\",\\n        licencia=\\\"CC BY 4.0\\\"),\\n    \\\"tomate_bangladesh_2025\\\": dict(\\n        tipo=\\\"mendeley\\\", id=\\\"93h9p62kg4\\\", version=1, campo=True, regla=\\\"tomate_bangladesh_2025\\\",\\n        modelos=[\\\"hortalizas\\\", \\\"router\\\"], requerido=False, activo=False,  # licencia no indicada en la p\u00e1gina\\n        cita=\\\"Tomato leaf diseases, Mendeley Data, 10.17632/93h9p62kg4.1\\\", licencia=\\\"no indicada (desactivado)\\\"),\\n    \\\"cafe_saposoa\\\": dict(\\n        tipo=\\\"mendeley\\\", id=\\\"mfpxg4y65r\\\", version=2, campo=True, regla=\\\"cafe\\\",\\n        modelos=[\\\"cafe\\\", \\\"router\\\"], requerido=True,\\n        cita=\\\"Coffee leaf dataset by phytosanitary class (Saposoa), Mendeley Data, 10.17632/mfpxg4y65r.2\\\",\\n        licencia=\\\"CC BY 4.0\\\"),\\n    \\\"cacao_monilia\\\": dict(\\n        tipo=\\\"url\\\", url=\\\"https://zenodo.org/records/17716661/files/CocoaMoniliaDataSet.zip?download=1\\\",\\n        extraer=[\\\"cocoapods_images/\\\", \\\"YOLO_annotations/\\\"], campo=True, regla=\\\"cacao_monilia\\\", recorte_yolo=True,\\n        modelos=[\\\"cacao\\\", \\\"router\\\"], requerido=True,\\n        cita=\\\"CocoaMoniliaDataSet, Zenodo, 10.5281/zenodo.17716661\\\", licencia=\\\"CC BY 4.0\\\"),\\n    \\\"cacao_kaggle\\\": dict(\\n        tipo=\\\"kaggle\\\", ref=\\\"serranosebas/enfermedades-cacao-yolov4\\\", campo=True, regla=\\\"cacao_yolo\\\",\\n        recorte_yolo=True, modelos=[\\\"cacao\\\", \\\"router\\\"], requerido=False,\\n        cita=\\\"Cocoa Diseases (YOLOv4), Kaggle serranosebas/enfermedades-cacao-yolov4\\\", licencia=\\\"ODbL\\\"),\\n    \\\"no_planta\\\": dict(\\n        tipo=\\\"kaggle\\\", ref=\\\"prasunroy/natural-images\\\", campo=True, regla=\\\"no_planta\\\",\\n        modelos=[\\\"router\\\"], requerido=True,\\n        cita=\\\"Roy et al. (2018), Natural Images, Kaggle prasunroy/natural-images\\\", licencia=\\\"ver la fuente\\\"),\\n}\\n\\n# ---------------------------------------------------------------- receta\\nRECETA = dict(\\n    backbones=[\\\"convnext_tiny.fb_in22k_ft_in1k\\\", \\\"tf_efficientnetv2_b2.in1k\\\"],\\n    tam=256,                 # entrada H = W\\n    lote=32,\\n    epocas_cabeza=2, lr_cabeza=1e-3,\\n    epocas=25, lr=1e-4, wd=0.05, suavizado=0.1, paciencia=6,\\n    muestras_por_epoca=12000,   # tope de fotos por \u00e9poca (el muestreo es con reemplazo)\\n    peso_campo=3.0,             # las fotos de campo salen 3 veces m\u00e1s en el muestreo\\n    max_por_clase_y_fuente=4000,\\n    tta=True,                   # espejo horizontal promediado dentro del ONNX\\n    precision_objetivo=0.90,    # el umbral se elige para acertar >= 90 % de lo que s\u00ed responde (en validaci\u00f3n)\\n    latencia_max_ms=300,        # backbone descartado si supera esto en CPU\\n    semilla=42,\\n)\\nRECETA_POR_MODELO = {\\n    \\\"router\\\": dict(muestras_por_epoca=16000, epocas=15),\\n    \\\"cafe\\\": dict(muestras_por_epoca=4000, epocas=30),\\n    \\\"cacao\\\": dict(muestras_por_epoca=4000, epocas=30),\\n}\\nMIN_FOTOS_CLASE = 30   # una clase con menos fotos de entrenamiento se quita del modelo (y se avisa)\\n\\n\\ndef receta(modelo):\\n    r = dict(RECETA)\\n    r.update(RECETA_POR_MODELO.get(modelo, {}))\\n    return r\\n\", \"datos.py\": \"\\\"\\\"\\\"Descarga, etiquetado, deduplicaci\u00f3n y partici\u00f3n de los datos.\\n\\nReglas de partici\u00f3n (las que hacen honesta la m\u00e9trica de campo):\\n- Solo las fotos de CAMPO entran en validaci\u00f3n y prueba. Las de laboratorio van a entrenamiento\\n  (y un 10 % a \\\"lab_prueba\\\", solo para informar el acierto en laboratorio).\\n- Las fotos casi id\u00e9nticas (mismo hash perceptual) forman un grupo y el grupo entero cae en una sola partici\u00f3n.\\n- La prueba no se usa para nada m\u00e1s que el informe final.\\n\\\"\\\"\\\"\\nimport csv, glob, hashlib, json, os, random, re, shutil, subprocess, zipfile\\nfrom collections import Counter, defaultdict\\nfrom multiprocessing import Pool\\n\\nfrom PIL import Image, ImageOps\\n\\nfrom . import catalogo, reglas\\n\\nEXT = (\\\".jpg\\\", \\\".jpeg\\\", \\\".png\\\", \\\".bmp\\\", \\\".webp\\\")\\nKAGGLE_INPUT = \\\"/kaggle/input\\\"\\nLADO_CACHE = 512\\n\\n\\ndef log(*a):\\n    print(\\\"[datos]\\\", *a, flush=True)\\n\\n\\n# ------------------------------------------------------------ descarga\\ndef _bajar(url, destino):\\n    os.makedirs(os.path.dirname(destino), exist_ok=True)\\n    subprocess.run([\\\"wget\\\", \\\"-q\\\", \\\"--tries=3\\\", \\\"-O\\\", destino, url], check=True)\\n\\n\\ndef _descomprimir(zip_path, carpeta, extraer=None):\\n    with zipfile.ZipFile(zip_path) as z:\\n        miembros = [m for m in z.namelist() if not extraer or any(p in m for p in extraer)]\\n        z.extractall(carpeta, members=miembros)\\n    os.remove(zip_path)\\n    for interno in glob.glob(os.path.join(carpeta, \\\"**\\\", \\\"*.zip\\\"), recursive=True):  # zips dentro de zips\\n        _descomprimir(interno, os.path.splitext(interno)[0])\\n\\n\\ndef _respaldo_kaggle(clave):\\n    for d in glob.glob(os.path.join(KAGGLE_INPUT, \\\"**\\\", f\\\"fitolens-{clave}*\\\"), recursive=True)[:1]:\\n        return d\\n    return None\\n\\n\\ndef obtener(clave, cfg, raiz):\\n    \\\"\\\"\\\"Devuelve la carpeta local con el dataset, descarg\u00e1ndolo si hace falta.\\\"\\\"\\\"\\n    destino = os.path.join(raiz, clave)\\n    if os.path.isdir(destino) and os.listdir(destino):\\n        return destino\\n    manual = _respaldo_kaggle(clave)\\n    if manual:\\n        log(f\\\"{clave}: usando la copia subida a Kaggle en {manual}\\\")\\n        return manual\\n    t = cfg[\\\"tipo\\\"]\\n    log(f\\\"{clave}: descargando ({t})\\\")\\n    if t == \\\"kaggle\\\":\\n        slug = cfg[\\\"ref\\\"].split(\\\"/\\\")[1]\\n        for d in glob.glob(os.path.join(KAGGLE_INPUT, \\\"**\\\", slug), recursive=True)[:1]:\\n            return d\\n        import kagglehub\\n        return kagglehub.dataset_download(cfg[\\\"ref\\\"])\\n    if t == \\\"git\\\":\\n        subprocess.run([\\\"git\\\", \\\"clone\\\", \\\"-q\\\", \\\"--depth\\\", \\\"1\\\", cfg[\\\"url\\\"], destino], check=True)\\n        return destino\\n    if t == \\\"mendeley\\\":\\n        url = f\\\"https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/{cfg['id']}-{cfg['version']}.zip\\\"\\n    else:\\n        url = cfg[\\\"url\\\"]\\n    z = destino + \\\".zip\\\"\\n    _bajar(url, z)\\n    _descomprimir(z, destino, cfg.get(\\\"extraer\\\"))\\n    return destino\\n\\n\\n# ------------------------------------------------------------ \u00edndice\\ndef _cajas_yolo(txt, nombres):\\n    \\\"\\\"\\\"Lee un .txt YOLO (cajas o pol\u00edgonos) y devuelve [(clase, x0, y0, x1, y1)] en coordenadas 0-1.\\\"\\\"\\\"\\n    cajas = []\\n    for linea in open(txt, encoding=\\\"utf-8\\\", errors=\\\"ignore\\\"):\\n        v = linea.split()\\n        if len(v) < 5:\\n            continue\\n        c, nums = int(float(v[0])), [float(x) for x in v[1:]]\\n        if len(nums) == 4:   # cx cy w h\\n            cx, cy, w, h = nums\\n            caja = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)\\n        else:                # pol\u00edgono x1 y1 x2 y2 ...\\n            xs, ys = nums[0::2], nums[1::2]\\n            caja = (min(xs), min(ys), max(xs), max(ys))\\n        cajas.append((nombres[c] if c < len(nombres) else str(c), *caja))\\n    return cajas\\n\\n\\ndef _nombres_yolo(carpeta):\\n    for patron in (\\\"**/*.names\\\", \\\"**/classes.txt\\\", \\\"**/_darknet.labels\\\"):\\n        for f in glob.glob(os.path.join(carpeta, patron), recursive=True):\\n            return [l.strip() for l in open(f, encoding=\\\"utf-8\\\") if l.strip()]\\n    for f in glob.glob(os.path.join(carpeta, \\\"**/data.yaml\\\"), recursive=True):\\n        m = re.search(r\\\"names:\\\\s*\\\\[(.*?)\\\\]\\\", open(f, encoding=\\\"utf-8\\\").read(), re.S)\\n        if m:\\n            return [x.strip(\\\" '\\\\\\\"\\\\n\\\") for x in m.group(1).split(\\\",\\\")]\\n    return []\\n\\n\\ndef indexar(clave, cfg, carpeta):\\n    regla = getattr(reglas, cfg[\\\"regla\\\"])\\n    imgs = [p for p in glob.glob(os.path.join(carpeta, \\\"**\\\", \\\"*\\\"), recursive=True) if p.lower().endswith(EXT)]\\n    if cfg.get(\\\"incluir\\\"):\\n        imgs = [p for p in imgs if cfg[\\\"incluir\\\"] in p.replace(\\\"\\\\\\\\\\\", \\\"/\\\").lower()]\\n    txts = {}\\n    nombres = []\\n    if cfg.get(\\\"recorte_yolo\\\"):\\n        nombres = _nombres_yolo(carpeta)\\n        for t in glob.glob(os.path.join(carpeta, \\\"**\\\", \\\"*.txt\\\"), recursive=True):\\n            txts.setdefault(os.path.splitext(os.path.basename(t))[0], t)\\n    muestras, resumen = [], Counter()\\n    for p in imgs:\\n        rel = os.path.relpath(p, carpeta).replace(\\\"\\\\\\\\\\\", \\\"/\\\")\\n        carp = os.path.basename(os.path.dirname(p))\\n        split = None\\n        if cfg.get(\\\"split_por_carpeta\\\"):\\n            split = \\\"test\\\" if \\\"/test/\\\" in f\\\"/{rel.lower()}\\\" else \\\"pool\\\"\\n        base = dict(ruta=p, fuente=clave, campo=cfg[\\\"campo\\\"], split_fijo=split, caja=None)\\n        txt = txts.get(os.path.splitext(os.path.basename(p))[0])\\n        if cfg.get(\\\"recorte_yolo\\\") and txt:\\n            for nom, *caja in _cajas_yolo(txt, nombres):\\n                et = regla(nom, cfg) if cfg[\\\"regla\\\"] == \\\"cacao_yolo\\\" else regla(carp, cfg)\\n                resumen[(nom if cfg[\\\"regla\\\"] == \\\"cacao_yolo\\\" else carp, et)] += 1\\n                if et and (caja[2] - caja[0]) * (caja[3] - caja[1]) > 0.002:\\n                    muestras.append(dict(base, etiqueta=et, caja=caja))\\n        else:\\n            et = regla(carp, cfg)\\n            resumen[(carp, et)] += 1\\n            if et:\\n                muestras.append(dict(base, etiqueta=et))\\n    log(f\\\"{clave}: {len(imgs)} im\u00e1genes, {len(muestras)} muestras. Carpeta -> etiqueta:\\\")\\n    for (c, e), n in sorted(resumen.items(), key=lambda x: -x[1])[:40]:\\n        log(f\\\"   {c!r:45} -> {e} ({n})\\\")\\n    return muestras\\n\\n\\n# ------------------------------------------------------------ cach\u00e9 redimensionada + hash perceptual\\ndef _cachear(args):\\n    m, carpeta_cache, con_hash = args\\n    clave = hashlib.md5((m[\\\"ruta\\\"] + str(m[\\\"caja\\\"])).encode()).hexdigest()\\n    destino = os.path.join(carpeta_cache, m[\\\"fuente\\\"], clave + \\\".jpg\\\")\\n    try:\\n        if not os.path.exists(destino):\\n            img = ImageOps.exif_transpose(Image.open(m[\\\"ruta\\\"])).convert(\\\"RGB\\\")\\n            if m[\\\"caja\\\"]:\\n                W, H = img.size\\n                x0, y0, x1, y1 = m[\\\"caja\\\"]\\n                mx, my = (x1 - x0) * 0.15, (y1 - y0) * 0.15\\n                img = img.crop((max(0, (x0 - mx) * W), max(0, (y0 - my) * H),\\n                                min(W, (x1 + mx) * W), min(H, (y1 + my) * H)))\\n            img.thumbnail((LADO_CACHE, LADO_CACHE), Image.BILINEAR)\\n            os.makedirs(os.path.dirname(destino), exist_ok=True)\\n            img.save(destino, quality=95)\\n        grupo = clave\\n        if con_hash:\\n            import imagehash\\n            grupo = str(imagehash.phash(Image.open(destino)))\\n        return destino, grupo\\n    except Exception as e:  # foto corrupta\\n        return None, str(e)\\n\\n\\ndef _partir(muestras, semilla):\\n    \\\"\\\"\\\"Asigna train / val / test / lab_prueba respetando grupos.\\\"\\\"\\\"\\n    rnd = random.Random(semilla)\\n    grupos = defaultdict(list)\\n    for m in muestras:\\n        grupos[(m[\\\"fuente\\\"], m[\\\"grupo\\\"])].append(m)\\n    asignado = {}\\n    por_fuente_et = defaultdict(list)\\n    for g, ms in grupos.items():\\n        m0 = ms[0]\\n        if not m0[\\\"campo\\\"]:\\n            asignado[g] = \\\"lab_prueba\\\" if rnd.random() < 0.10 else \\\"train\\\"\\n        elif m0[\\\"split_fijo\\\"] == \\\"test\\\":\\n            asignado[g] = \\\"test\\\"\\n        else:\\n            por_fuente_et[(m0[\\\"fuente\\\"], m0[\\\"etiqueta\\\"], m0[\\\"split_fijo\\\"])].append(g)\\n    for (fuente, et, fijo), gs in por_fuente_et.items():\\n        rnd.shuffle(gs)\\n        n = len(gs)\\n        if fijo == \\\"pool\\\":            # PlantDoc ya trae su \\\"test\\\": del resto, 15 % a validaci\u00f3n\\n            n_val, n_test = round(n * 0.15), 0\\n        else:                         # 70 / 15 / 15\\n            n_val = n_test = round(n * 0.15)\\n        for i, g in enumerate(gs):\\n            asignado[g] = \\\"test\\\" if i < n_test else \\\"val\\\" if i < n_test + n_val else \\\"train\\\"\\n    # una foto repetida en dos fuentes: prueba > validaci\u00f3n > entrenamiento\\n    prioridad = {\\\"test\\\": 3, \\\"val\\\": 2, \\\"lab_prueba\\\": 1, \\\"train\\\": 0}\\n    mejor = {}\\n    for (fuente, grupo), sp in asignado.items():\\n        if prioridad[sp] > prioridad.get(mejor.get(grupo), -1):\\n            mejor[grupo] = sp\\n    for m in muestras:\\n        sp = asignado[(m[\\\"fuente\\\"], m[\\\"grupo\\\"])]\\n        m[\\\"split\\\"] = sp if mejor[m[\\\"grupo\\\"]] == sp else None   # copia en otra partici\u00f3n: se descarta\\n    return [m for m in muestras if m[\\\"split\\\"]]\\n\\n\\ndef preparar(modelo, trabajo, semilla=42, prueba=False):\\n    \\\"\\\"\\\"Devuelve (muestras, clases, info). Cada muestra: ruta (cach\u00e9), y (\u00edndice), campo, split, fuente.\\\"\\\"\\\"\\n    clases = list(catalogo.CLASES[modelo])\\n    indice_csv = os.path.join(trabajo, f\\\"indice_{modelo}.csv\\\")\\n    if prueba:\\n        muestras = _datos_falsos(clases, os.path.join(trabajo, \\\"falsos\\\"))\\n    elif os.path.exists(indice_csv):\\n        log(f\\\"reutilizando {indice_csv}\\\")\\n        muestras = [dict(r, campo=r[\\\"campo\\\"] == \\\"True\\\") for r in csv.DictReader(open(indice_csv, encoding=\\\"utf-8\\\"))]\\n    else:\\n        muestras, fallos = [], []\\n        for clave, cfg in catalogo.FUENTES.items():\\n            if modelo not in cfg[\\\"modelos\\\"] or not cfg.get(\\\"activo\\\", True):\\n                continue\\n            try:\\n                carpeta = obtener(clave, cfg, os.path.join(trabajo, \\\"fuentes\\\"))\\n                ms = indexar(clave, cfg, carpeta)\\n            except Exception as e:\\n                if cfg[\\\"requerido\\\"]:\\n                    raise RuntimeError(f\\\"No se pudo obtener '{clave}' ({e}). S\u00fabelo a Kaggle como \\\"\\n                                       f\\\"'fitolens-{clave}' y agr\u00e9galo al notebook.\\\") from e\\n                log(f\\\"AVISO: se omite {clave}: {e}\\\")\\n                fallos.append(clave)\\n                continue\\n            rnd = random.Random(semilla)\\n            por_et = defaultdict(list)\\n            for m in ms:\\n                et = reglas.etiqueta_para_modelo(m[\\\"etiqueta\\\"], modelo, clases)\\n                if et:\\n                    por_et[et].append(dict(m, etiqueta=et))\\n            for et, lista in por_et.items():\\n                rnd.shuffle(lista)\\n                muestras += lista[: catalogo.receta(modelo)[\\\"max_por_clase_y_fuente\\\"]]\\n        log(f\\\"cacheando {len(muestras)} fotos (redimensi\u00f3n a {LADO_CACHE}px y hash perceptual)\\\")\\n        cache = os.path.join(trabajo, \\\"cache\\\")\\n        with Pool(os.cpu_count()) as pool:\\n            res = pool.map(_cachear, [(m, cache, m[\\\"campo\\\"]) for m in muestras], chunksize=64)\\n        ok = []\\n        for m, (ruta, grupo) in zip(muestras, res):\\n            if ruta:\\n                ok.append(dict(m, ruta=ruta, grupo=grupo))\\n        log(f\\\"{len(muestras) - len(ok)} fotos ilegibles descartadas\\\")\\n        muestras = _partir(ok, semilla)\\n        with open(indice_csv, \\\"w\\\", newline=\\\"\\\", encoding=\\\"utf-8\\\") as f:\\n            campos = [\\\"ruta\\\", \\\"etiqueta\\\", \\\"fuente\\\", \\\"campo\\\", \\\"split\\\", \\\"grupo\\\"]\\n            w = csv.DictWriter(f, fieldnames=campos, extrasaction=\\\"ignore\\\")\\n            w.writeheader()\\n            w.writerows(muestras)\\n    # clases sin suficientes fotos de entrenamiento: se quitan y se avisa\\n    n_train = Counter(m[\\\"etiqueta\\\"] for m in muestras if m[\\\"split\\\"] == \\\"train\\\")\\n    quitadas = [c for c in clases if n_train[c] < catalogo.MIN_FOTOS_CLASE]\\n    if quitadas:\\n        log(f\\\"AVISO: clases quitadas por tener < {catalogo.MIN_FOTOS_CLASE} fotos de entrenamiento: {quitadas}\\\")\\n        clases = [c for c in clases if c not in quitadas]\\n    pos = {c: i for i, c in enumerate(clases)}\\n    muestras = [dict(m, y=pos[m[\\\"etiqueta\\\"]]) for m in muestras if m[\\\"etiqueta\\\"] in pos]\\n    tabla = defaultdict(Counter)\\n    for m in muestras:\\n        tabla[m[\\\"etiqueta\\\"]][m[\\\"split\\\"]] += 1\\n    log(\\\"fotos por clase y partici\u00f3n (val y test son solo de campo):\\\")\\n    for c in clases:\\n        t = tabla[c]\\n        log(f\\\"   {c:32} train={t['train']:6} val={t['val']:5} test={t['test']:5} lab_prueba={t['lab_prueba']:5}\\\")\\n    fuentes = sorted({m[\\\"fuente\\\"] for m in muestras})\\n    info = dict(conteos={c: dict(tabla[c]) for c in clases}, clases_quitadas=quitadas,\\n                datasets=[dict(clave=f, cita=catalogo.FUENTES.get(f, {}).get(\\\"cita\\\", f),\\n                               licencia=catalogo.FUENTES.get(f, {}).get(\\\"licencia\\\", \\\"\\\")) for f in fuentes])\\n    return muestras, clases, info\\n\\n\\ndef _datos_falsos(clases, carpeta):\\n    \\\"\\\"\\\"Fotos sint\u00e9ticas para probar el pipeline sin descargar nada (modo --prueba).\\\"\\\"\\\"\\n    import numpy as np\\n    rnd = np.random.default_rng(0)\\n    muestras = []\\n    for i, c in enumerate(clases):\\n        for j in range(60):\\n            ruta = os.path.join(carpeta, c, f\\\"{j}.jpg\\\")\\n            if not os.path.exists(ruta):\\n                os.makedirs(os.path.dirname(ruta), exist_ok=True)\\n                base = np.full((96, 96, 3), (i * 37) % 255, np.uint8)\\n                base = np.clip(base + rnd.integers(0, 60, base.shape), 0, 255).astype(np.uint8)\\n                Image.fromarray(base).save(ruta)\\n            sp = \\\"train\\\" if j < 40 else \\\"val\\\" if j < 50 else \\\"test\\\"\\n            muestras.append(dict(ruta=ruta, etiqueta=c, fuente=\\\"falsa\\\", campo=j % 2 == 0 or sp != \\\"train\\\",\\n                                 split=sp, grupo=f\\\"{c}{j}\\\"))\\n    return muestras\\n\", \"entrenar.py\": \"\\\"\\\"\\\"Entrena un modelo de FitoLens de principio a fin y deja model.onnx + manifest.json + metricas.json.\\n\\nUso:  python -m fitolens_ent.entrenar --modelo hortalizas        (router | hortalizas | cafe | cacao)\\n      python -m fitolens_ent.entrenar --modelo cafe --prueba    (prueba r\u00e1pida con datos falsos, en CPU)\\n\\\"\\\"\\\"\\nimport argparse, copy, datetime, hashlib, json, math, os, random, shutil, time\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn as nn\\nfrom torch.utils.data import DataLoader, WeightedRandomSampler\\n\\nfrom . import catalogo, datos\\nfrom .aumentos import Conjunto\\nfrom .modelo import FitoNet, ParaExportar, exportar_onnx\\n\\nDISP = \\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\"\\n\\n\\ndef log(*a):\\n    print(\\\"[entrenar]\\\", *a, flush=True)\\n\\n\\n# ------------------------------------------------------------ m\u00e9tricas\\ndef probabilidades(logits, T, grupos=None, y=None):\\n    \\\"\\\"\\\"softmax(logits/T). En hortalizas, igual que la API: solo entre las clases del cultivo (aqu\u00ed, el real).\\\"\\\"\\\"\\n    z = logits / T\\n    if grupos is not None:\\n        mascara = grupos[None, :] != grupos[y][:, None]\\n        z = np.where(mascara, -1e9, z)\\n    z = z - z.max(1, keepdims=True)\\n    p = np.exp(z)\\n    return p / p.sum(1, keepdims=True)\\n\\n\\ndef wilson(aciertos, n, z=1.96):\\n    if n == 0:\\n        return [0.0, 0.0]\\n    p = aciertos / n\\n    c = (p + z * z / (2 * n)) / (1 + z * z / n)\\n    d = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)\\n    return [round(c - d, 4), round(c + d, 4)]\\n\\n\\ndef metricas(p, y, clases, umbral=None):\\n    pred, conf = p.argmax(1), p.max(1)\\n    K, n = len(clases), len(y)\\n    mc = np.zeros((K, K), int)\\n    for a, b in zip(y, pred):\\n        mc[a, b] += 1\\n    por_clase, f1s = {}, []\\n    for k, c in enumerate(clases):\\n        tp, sop, prd = mc[k, k], mc[k].sum(), mc[:, k].sum()\\n        prec = tp / prd if prd else 0.0\\n        rec = tp / sop if sop else 0.0\\n        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0\\n        por_clase[c] = dict(n=int(sop), precision=round(prec, 4), recall=round(rec, 4), f1=round(f1, 4))\\n        if sop:\\n            f1s.append(f1)\\n    acc = float((pred == y).mean()) if n else 0.0\\n    bins = np.linspace(0, 1, 11)\\n    ece = sum(abs((pred[s] == y[s]).mean() - conf[s].mean()) * s.mean()\\n              for s in [(conf > a) & (conf <= b) for a, b in zip(bins[:-1], bins[1:])] if s.any())\\n    r = dict(n=int(n), accuracy=round(acc, 4), accuracy_ic95=wilson(int((pred == y).sum()), n),\\n             f1_macro=round(float(np.mean(f1s)) if f1s else 0.0, 4), ece=round(float(ece), 4),\\n             por_clase=por_clase, matriz_confusion=mc.tolist(),\\n             clases_sin_fotos=[c for c in clases if por_clase[c][\\\"n\\\"] == 0])\\n    if umbral is not None:\\n        resp = conf >= umbral\\n        r[\\\"con_umbral\\\"] = dict(umbral=umbral, cobertura=round(float(resp.mean()), 4),\\n                               accuracy_respondidas=round(float((pred[resp] == y[resp]).mean()) if resp.any() else 0.0, 4))\\n    return r\\n\\n\\ndef elegir_temperatura(logits, y, grupos):\\n    mejor = (1e9, 1.0)\\n    for T in np.linspace(0.5, 5.0, 46):\\n        p = probabilidades(logits, T, grupos, y)\\n        nll = -np.log(p[np.arange(len(y)), y] + 1e-12).mean()\\n        mejor = min(mejor, (nll, float(T)))\\n    return round(mejor[1], 2)\\n\\n\\ndef elegir_umbral(p, y, objetivo):\\n    pred, conf = p.argmax(1), p.max(1)\\n    opciones = []\\n    for t in np.arange(0.30, 0.96, 0.01):\\n        resp = conf >= t\\n        if resp.mean() < 0.3:\\n            break\\n        opciones.append((round(float(t), 2), float((pred[resp] == y[resp]).mean()), float(resp.mean())))\\n    for t, acc, cob in opciones:\\n        if acc >= objetivo:\\n            return t, True\\n    return (max(opciones, key=lambda o: o[1])[0] if opciones else 0.5), False\\n\\n\\n# ------------------------------------------------------------ entrenamiento\\n@torch.no_grad()\\ndef predecir(red, cargador, tta=True):\\n    red.eval()\\n    L, Y = [], []\\n    for x, y in cargador:\\n        x = x.to(DISP, non_blocking=True).to(memory_format=torch.channels_last)\\n        with torch.autocast(DISP, enabled=DISP == \\\"cuda\\\"):\\n            l = red(x)\\n            if tta:\\n                l = (l + red(torch.flip(x, dims=[3]))) / 2\\n        L.append(l.float().cpu())\\n        Y.append(y)\\n    return torch.cat(L).numpy(), torch.cat(Y).numpy()\\n\\n\\ndef entrenar_backbone(backbone, muestras, clases, r, grupos, prueba):\\n    tr = [m for m in muestras if m[\\\"split\\\"] == \\\"train\\\"]\\n    va = [m for m in muestras if m[\\\"split\\\"] == \\\"val\\\"]\\n    fondos = [m[\\\"ruta\\\"] for m in tr if m[\\\"campo\\\"]][:3000]\\n    cuenta = np.bincount([m[\\\"y\\\"] for m in tr], minlength=len(clases))\\n    pesos = [(1 / math.sqrt(cuenta[m[\\\"y\\\"]])) * (r[\\\"peso_campo\\\"] if m[\\\"campo\\\"] else 1.0) for m in tr]\\n    nw = min(8, os.cpu_count() or 2)\\n    ctr = DataLoader(Conjunto(tr, r[\\\"tam\\\"], True, fondos), r[\\\"lote\\\"], num_workers=nw, pin_memory=True,\\n                     sampler=WeightedRandomSampler(pesos, r[\\\"muestras_por_epoca\\\"], replacement=True),\\n                     drop_last=True, persistent_workers=nw > 0)\\n    cva = DataLoader(Conjunto(va, r[\\\"tam\\\"]), r[\\\"lote\\\"] * 2, num_workers=nw)\\n\\n    red = FitoNet(backbone, len(clases), preentrenado=not prueba).to(DISP).to(memory_format=torch.channels_last)\\n    criterio = nn.CrossEntropyLoss(label_smoothing=r[\\\"suavizado\\\"])\\n    escalador = torch.amp.GradScaler(enabled=DISP == \\\"cuda\\\")\\n\\n    def una_epoca(opt, sched=None):\\n        red.train()\\n        total, t0 = 0.0, time.time()\\n        for x, y in ctr:\\n            x = x.to(DISP, non_blocking=True).to(memory_format=torch.channels_last)\\n            y = y.to(DISP)\\n            with torch.autocast(DISP, enabled=DISP == \\\"cuda\\\"):\\n                perdida = criterio(red(x), y)\\n            opt.zero_grad(set_to_none=True)\\n            escalador.scale(perdida).backward()\\n            escalador.unscale_(opt)\\n            nn.utils.clip_grad_norm_(red.parameters(), 2.0)\\n            escalador.step(opt)\\n            escalador.update()\\n            if sched:\\n                sched.step()\\n            total += perdida.item()\\n        return total / max(1, len(ctr)), time.time() - t0\\n\\n    def f1_val():\\n        l, y = predecir(red, cva, tta=False)\\n        return metricas(probabilidades(l, 1.0, grupos, y), y, clases)[\\\"f1_macro\\\"]\\n\\n    # fase 1: solo la cabeza\\n    for p in red.base.parameters():\\n        p.requires_grad = False\\n    for p in red.cabeza():\\n        p.requires_grad = True\\n    opt = torch.optim.AdamW(red.cabeza(), lr=r[\\\"lr_cabeza\\\"], weight_decay=1e-4)\\n    for e in range(r[\\\"epocas_cabeza\\\"]):\\n        perd, s = una_epoca(opt)\\n        log(f\\\"{backbone} cabeza {e + 1}/{r['epocas_cabeza']} p\u00e9rdida={perd:.3f} f1_val={f1_val():.4f} ({s:.0f}s)\\\")\\n\\n    # fase 2: ajuste fino completo, coseno con calentamiento\\n    for p in red.parameters():\\n        p.requires_grad = True\\n    opt = torch.optim.AdamW(red.parameters(), lr=r[\\\"lr\\\"], weight_decay=r[\\\"wd\\\"])\\n    pasos = r[\\\"epocas\\\"] * len(ctr)\\n    cal = max(1, int(0.05 * pasos))\\n    sched = torch.optim.lr_scheduler.LambdaLR(\\n        opt, lambda i: (i + 1) / cal if i < cal else 0.5 * (1 + math.cos(math.pi * (i - cal) / max(1, pasos - cal))))\\n    mejor, estado, sin_mejora = -1.0, None, 0\\n    for e in range(r[\\\"epocas\\\"]):\\n        perd, s = una_epoca(opt, sched)\\n        f1 = f1_val()\\n        log(f\\\"{backbone} \u00e9poca {e + 1}/{r['epocas']} p\u00e9rdida={perd:.3f} f1_val_campo={f1:.4f} ({s:.0f}s)\\\")\\n        if f1 > mejor:\\n            mejor, estado, sin_mejora = f1, copy.deepcopy(red.state_dict()), 0\\n        else:\\n            sin_mejora += 1\\n            if sin_mejora >= r[\\\"paciencia\\\"]:\\n                log(\\\"parada temprana\\\")\\n                break\\n    red.load_state_dict(estado)\\n    return red, mejor, cva\\n\\n\\ndef latencia_y_paridad(ruta_onnx, red, cva, tam):\\n    import onnxruntime as ort\\n    so = ort.SessionOptions()\\n    so.intra_op_num_threads = min(4, os.cpu_count() or 1)\\n    ses = ort.InferenceSession(ruta_onnx, so, providers=[\\\"CPUExecutionProvider\\\"])\\n    x, _ = next(iter(cva))\\n    x = x[:4]\\n    ref = ParaExportar(copy.deepcopy(red).cpu().float(), True)(x)[0].detach().numpy()\\n    out = ses.run(None, {\\\"imagen\\\": x.numpy()})\\n    dif = float(np.abs(out[0] - ref).max())\\n    uno = x[:1].numpy()\\n    for _ in range(5):\\n        ses.run(None, {\\\"imagen\\\": uno})\\n    tiempos = []\\n    for _ in range(30):\\n        t0 = time.perf_counter()\\n        ses.run(None, {\\\"imagen\\\": uno})\\n        tiempos.append((time.perf_counter() - t0) * 1000)\\n    return round(float(np.median(tiempos)), 1), dif, list(out[1].shape[2:])\\n\\n\\ndef sha256(ruta):\\n    h = hashlib.sha256()\\n    with open(ruta, \\\"rb\\\") as f:\\n        for b in iter(lambda: f.read(1 << 20), b\\\"\\\"):\\n            h.update(b)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--modelo\\\", required=True, choices=list(catalogo.CLASES))\\n    ap.add_argument(\\\"--trabajo\\\", default=\\\"/kaggle/working/fitolens\\\")\\n    ap.add_argument(\\\"--salida\\\", default=\\\"/kaggle/working/salida\\\")\\n    ap.add_argument(\\\"--backbones\\\", nargs=\\\"*\\\")\\n    ap.add_argument(\\\"--version\\\", default=datetime.date.today().strftime(\\\"%Y.%m.%d\\\"))\\n    ap.add_argument(\\\"--prueba\\\", action=\\\"store_true\\\", help=\\\"datos falsos y red m\u00ednima, para probar el pipeline\\\")\\n    a = ap.parse_args()\\n\\n    r = catalogo.receta(a.modelo)\\n    if a.prueba:\\n        r.update(backbones=[\\\"resnet18\\\"], tam=64, lote=8, epocas_cabeza=1, epocas=1, muestras_por_epoca=64)\\n    if a.backbones:\\n        r[\\\"backbones\\\"] = a.backbones\\n    random.seed(r[\\\"semilla\\\"]); np.random.seed(r[\\\"semilla\\\"]); torch.manual_seed(r[\\\"semilla\\\"])\\n    os.makedirs(a.trabajo, exist_ok=True)\\n    log(f\\\"modelo={a.modelo} dispositivo={DISP} receta={r}\\\")\\n\\n    muestras, clases, info = datos.preparar(a.modelo, a.trabajo, r[\\\"semilla\\\"], a.prueba)\\n    if not any(m[\\\"split\\\"] == \\\"val\\\" for m in muestras) or not any(m[\\\"split\\\"] == \\\"test\\\" for m in muestras):\\n        raise RuntimeError(\\\"No hay fotos de campo para validaci\u00f3n o prueba: revisa las fuentes.\\\")\\n    grupos = None\\n    if a.modelo == \\\"hortalizas\\\":\\n        cult = sorted({c.split(\\\"_\\\")[0] for c in clases})\\n        grupos = np.array([cult.index(c.split(\\\"_\\\")[0]) for c in clases])\\n\\n    tmp = os.path.join(a.trabajo, \\\"candidatos\\\")\\n    os.makedirs(tmp, exist_ok=True)\\n    candidatos = []\\n    for bb in r[\\\"backbones\\\"]:\\n        red, f1_sin_tta, cva = entrenar_backbone(bb, muestras, clases, r, grupos, a.prueba)\\n        lv, yv = predecir(red, cva, r[\\\"tta\\\"])\\n        T = elegir_temperatura(lv, yv, grupos)\\n        pv = probabilidades(lv, T, grupos, yv)\\n        umbral, alcanzado = elegir_umbral(pv, yv, r[\\\"precision_objetivo\\\"])\\n        f1v = metricas(pv, yv, clases)[\\\"f1_macro\\\"]\\n        ruta = os.path.join(tmp, bb.replace(\\\"/\\\", \\\"_\\\") + \\\".onnx\\\")\\n        exportar_onnx(copy.deepcopy(red).cpu().float(), ruta, r[\\\"tam\\\"], r[\\\"tta\\\"])\\n        lat, dif, cam_hw = latencia_y_paridad(ruta, red, cva, r[\\\"tam\\\"])\\n        log(f\\\"{bb}: f1_val_campo={f1v:.4f} T={T} umbral={umbral} latencia_cpu={lat}ms dif_onnx={dif:.2e}\\\")\\n        if dif > 1e-2:\\n            log(\\\"AVISO: el ONNX difiere de PyTorch m\u00e1s de 1e-2\\\")\\n        candidatos.append(dict(backbone=bb, red=red.cpu(), f1_val=f1v, T=T, umbral=umbral, alcanzado=alcanzado,\\n                               onnx=ruta, latencia_ms=lat, dif_onnx=dif, cam_hw=cam_hw))\\n        torch.cuda.empty_cache()\\n\\n    validos = [c for c in candidatos if c[\\\"latencia_ms\\\"] <= r[\\\"latencia_max_ms\\\"]] or candidatos\\n    c = max(validos, key=lambda c: c[\\\"f1_val\\\"])      # se elige con VALIDACI\u00d3N, nunca con la prueba\\n    log(f\\\"elegido: {c['backbone']}\\\")\\n\\n    # informe final con la prueba de campo (no se toc\u00f3 antes)\\n    red = c[\\\"red\\\"].to(DISP)\\n    conj = lambda sp: DataLoader(Conjunto([m for m in muestras if m[\\\"split\\\"] == sp], r[\\\"tam\\\"]), r[\\\"lote\\\"] * 2,\\n                                 num_workers=min(8, os.cpu_count() or 2))\\n    lt, yt = predecir(red, conj(\\\"test\\\"), r[\\\"tta\\\"])\\n    campo = metricas(probabilidades(lt, c[\\\"T\\\"], grupos, yt), yt, clases, c[\\\"umbral\\\"])\\n    campo_sin_filtro = metricas(probabilidades(lt, c[\\\"T\\\"]), yt, clases) if grupos is not None else None\\n    lab = None\\n    if any(m[\\\"split\\\"] == \\\"lab_prueba\\\" for m in muestras):\\n        ll, yl = predecir(red, conj(\\\"lab_prueba\\\"), r[\\\"tta\\\"])\\n        lab = metricas(probabilidades(ll, c[\\\"T\\\"], grupos, yl), yl, clases)\\n    log(f\\\"PRUEBA DE CAMPO: n={campo['n']} accuracy={campo['accuracy']} IC95={campo['accuracy_ic95']} \\\"\\n        f\\\"f1_macro={campo['f1_macro']} | con umbral {c['umbral']}: {campo['con_umbral']}\\\")\\n\\n    # carpeta final con el contrato de la API\\n    dest = os.path.join(a.salida, a.modelo)\\n    shutil.rmtree(dest, ignore_errors=True)\\n    os.makedirs(dest)\\n    shutil.copy(c[\\\"onnx\\\"], os.path.join(dest, \\\"model.onnx\\\"))\\n    manifest = dict(\\n        nombre=a.modelo, version=a.version, tipo=\\\"router\\\" if a.modelo == \\\"router\\\" else \\\"especialista\\\",\\n        backbone=c[\\\"backbone\\\"],\\n        clases=[dict(id=k, nombre=n, etiqueta=catalogo.ETIQUETAS.get(n, n),\\n                     cultivo=n if a.modelo == \\\"router\\\" else n.split(\\\"_\\\")[0],\\n                     sana=n.endswith(\\\"_sana\\\")) for k, n in enumerate(clases)],\\n        entrada=dict(alto=r[\\\"tam\\\"], ancho=r[\\\"tam\\\"], formato=\\\"RGB\\\", rango=[0, 1], redimension=\\\"directa\\\",\\n                     interpolacion=\\\"bilinear\\\", orientacion_exif=True, layout=\\\"NCHW\\\"),\\n        salidas=dict(logits=\\\"logits\\\", cam=\\\"cam\\\", cam_tam=c[\\\"cam_hw\\\"], tta_incluido=r[\\\"tta\\\"]),\\n        temperatura=c[\\\"T\\\"], umbral=c[\\\"umbral\\\"], sha256=sha256(os.path.join(dest, \\\"model.onnx\\\")),\\n        entrenado=dict(fecha=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec=\\\"seconds\\\"),\\n                       datasets=info[\\\"datasets\\\"], receta={k: v for k, v in r.items() if k != \\\"backbones\\\"}),\\n    )\\n    if a.modelo == \\\"router\\\":\\n        manifest[\\\"rechazo\\\"] = catalogo.RECHAZO\\n        manifest[\\\"especialistas\\\"] = catalogo.ESPECIALISTA_DE_CULTIVO\\n    metr = dict(\\n        modelo=a.modelo, version=a.version, backbone=c[\\\"backbone\\\"],\\n        campo=campo, laboratorio=dict(accuracy=lab[\\\"accuracy\\\"], n=lab[\\\"n\\\"]) if lab else None,\\n        latencia_ms=c[\\\"latencia_ms\\\"],\\n        nota=(\\\"Prueba solo con fotos de campo que no se usaron para entrenar ni para elegir \u00e9pocas, \\\"\\n              \\\"temperatura, umbral ni backbone.\\\" +\\n              (\\\" En hortalizas las probabilidades se calculan solo entre las clases del cultivo, como hace la API \\\"\\n               \\\"cuando el router acierta el cultivo.\\\" if grupos is not None else \\\"\\\")),\\n        campo_sin_filtro_cultivo=campo_sin_filtro,\\n        validacion=dict(f1_macro=c[\\\"f1_val\\\"], precision_objetivo=r[\\\"precision_objetivo\\\"],\\n                        precision_objetivo_alcanzada=c[\\\"alcanzado\\\"]),\\n        candidatos=[{k: v for k, v in x.items() if k not in (\\\"red\\\", \\\"onnx\\\")} for x in candidatos],\\n        datos=dict(conteos=info[\\\"conteos\\\"], clases_quitadas=info[\\\"clases_quitadas\\\"]),\\n    )\\n    json.dump(manifest, open(os.path.join(dest, \\\"manifest.json\\\"), \\\"w\\\", encoding=\\\"utf-8\\\"), ensure_ascii=False, indent=2)\\n    json.dump(metr, open(os.path.join(dest, \\\"metricas.json\\\"), \\\"w\\\", encoding=\\\"utf-8\\\"), ensure_ascii=False, indent=2)\\n    zip_ = shutil.make_archive(os.path.join(a.salida, f\\\"modelo_{a.modelo}\\\"), \\\"zip\\\", a.salida, a.modelo)\\n    log(f\\\"listo: {dest}  ->  descarga {zip_}\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"modelo.py\": \"\\\"\\\"\\\"Red de FitoLens: backbone timm + cabeza GAP -> Linear, que permite el mapa de calor (CAM) exacto.\\n\\nLa normalizaci\u00f3n ImageNet va dentro de la red: la entrada siempre es RGB en rango 0-1.\\n\\\"\\\"\\\"\\nimport torch\\nimport torch.nn as nn\\nimport timm\\n\\nMEDIA = (0.485, 0.456, 0.406)\\nDESV = (0.229, 0.224, 0.225)\\n\\n\\nclass FitoNet(nn.Module):\\n    def __init__(self, backbone, n_clases, preentrenado=True):\\n        super().__init__()\\n        base = timm.create_model(backbone, pretrained=preentrenado, drop_path_rate=0.1)\\n        cfg = base.pretrained_cfg or {}\\n        media, desv = cfg.get(\\\"mean\\\", MEDIA), cfg.get(\\\"std\\\", DESV)\\n        self.register_buffer(\\\"media\\\", torch.tensor(media).view(1, 3, 1, 1))\\n        self.register_buffer(\\\"desv\\\", torch.tensor(desv).view(1, 3, 1, 1))\\n        # ConvNeXt normaliza despu\u00e9s del pooling (LayerNorm2d): se aplica por posici\u00f3n y el CAM sigue exacto\\n        norm = getattr(getattr(base, \\\"head\\\", None), \\\"norm\\\", None)\\n        self.norm = norm if isinstance(norm, nn.Module) else nn.Identity()\\n        base.reset_classifier(0, \\\"\\\")\\n        self.base = base\\n        self.drop = nn.Dropout(0.3)\\n        self.fc = nn.Linear(base.num_features, n_clases)\\n\\n    def mapas(self, x01):\\n        return self.norm(self.base.forward_features((x01 - self.media) / self.desv))\\n\\n    def forward(self, x01):\\n        return self.fc(self.drop(self.mapas(x01).mean((2, 3))))\\n\\n    def cabeza(self):\\n        return list(self.fc.parameters()) + list(self.norm.parameters())\\n\\n\\nclass ParaExportar(nn.Module):\\n    \\\"\\\"\\\"Lo que va dentro del ONNX: logits (con TTA de espejo) y CAM por clase.\\\"\\\"\\\"\\n\\n    def __init__(self, red, tta=True):\\n        super().__init__()\\n        self.red, self.tta = red.eval(), tta\\n\\n    def forward(self, imagen):\\n        f = self.red.mapas(imagen)\\n        logits = self.red.fc(f.mean((2, 3)))\\n        if self.tta:\\n            logits = (logits + self.red.fc(self.red.mapas(torch.flip(imagen, dims=[3])).mean((2, 3)))) / 2\\n        cam = torch.einsum(\\\"kc,nchw->nkhw\\\", self.red.fc.weight, f)\\n        return logits, cam\\n\\n\\ndef exportar_onnx(red, ruta, tam, tta=True):\\n    m = ParaExportar(red, tta).cpu().eval()\\n    x = torch.rand(1, 3, tam, tam)\\n    kw = dict(input_names=[\\\"imagen\\\"], output_names=[\\\"logits\\\", \\\"cam\\\"], opset_version=17,\\n              dynamic_axes={\\\"imagen\\\": {0: \\\"n\\\"}, \\\"logits\\\": {0: \\\"n\\\"}, \\\"cam\\\": {0: \\\"n\\\"}})\\n    try:\\n        torch.onnx.export(m, (x,), ruta, dynamo=False, **kw)\\n    except TypeError:   # versiones de torch sin el par\u00e1metro dynamo\\n        torch.onnx.export(m, (x,), ruta, **kw)\\n    return m\\n\", \"reglas.py\": \"\\\"\\\"\\\"Reglas que convierten el nombre de carpeta (o de clase YOLO) de cada dataset en una etiqueta FitoLens.\\n\\nDevuelven:\\n- una clase de enfermedad (p. ej. \\\"tomate_tizon_tardio\\\"),\\n- solo el cultivo (p. ej. \\\"tomate\\\"): la foto sirve para el router, no para el especialista,\\n- \\\"otra_planta\\\" / \\\"no_planta\\\", o\\n- None: la foto se ignora.\\nEl resumen \\\"carpeta -> etiqueta\\\" se imprime al preparar los datos para poder revisarlo.\\n\\\"\\\"\\\"\\nimport re\\n\\nCULTIVOS = (\\\"pimiento\\\", \\\"papa\\\", \\\"tomate\\\", \\\"cafe\\\", \\\"cacao\\\")\\n\\n\\ndef _norm(s):\\n    return re.sub(r\\\"[_\\\\-,.()]+\\\", \\\" \\\", s.lower()).strip()\\n\\n\\ndef _cultivo(s):\\n    if re.search(r\\\"pepper|pimiento|capsicum\\\", s):\\n        return \\\"pimiento\\\"\\n    if re.search(r\\\"potato|papa\\\\b\\\", s):\\n        return \\\"papa\\\"\\n    if re.search(r\\\"tomato|tomate\\\", s):\\n        return \\\"tomate\\\"\\n    return None\\n\\n\\ndef _enfermedad(s, cultivo):\\n    # el orden importa: lo m\u00e1s espec\u00edfico primero\\n    if \\\"septoria\\\" in s:\\n        return \\\"septoriosis\\\"\\n    if \\\"target\\\" in s:\\n        return \\\"mancha_diana\\\"\\n    if re.search(r\\\"yellow|curl\\\", s):\\n        return \\\"virus_rizado_amarillo\\\"\\n    if \\\"mosaic\\\" in s:\\n        return \\\"virus_mosaico\\\"\\n    if re.search(r\\\"spider|mite\\\", s):\\n        return \\\"arana_roja\\\"\\n    if re.search(r\\\"mold|mould\\\", s):\\n        return \\\"moho_hoja\\\"\\n    if \\\"early\\\" in s:\\n        return \\\"tizon_temprano\\\"\\n    if re.search(r\\\"late|phytophthora\\\", s):\\n        return \\\"tizon_tardio\\\"\\n    if re.search(r\\\"bacteri|spot\\\", s):\\n        return \\\"mancha_bacteriana\\\"\\n    if re.search(r\\\"healthy|sana|sano\\\", s):\\n        return \\\"sana\\\"\\n    return None\\n\\n\\ndef hortalizas(carpeta, cfg):\\n    s = _norm(carpeta)\\n    c = _cultivo(s)\\n    if c is None:\\n        return \\\"otra_planta\\\"\\n    e = _enfermedad(s, c)\\n    if e is None and cfg.get(\\\"sin_enfermedad_es_sana\\\"):\\n        e = \\\"sana\\\"   # PlantDoc: \\\"Tomato leaf\\\" y \\\"Bell_pepper leaf\\\" son hojas sanas\\n    return f\\\"{c}_{e}\\\" if e else c\\n\\n\\ndef tomate_bangladesh_2025(carpeta, cfg):\\n    s = _norm(carpeta)\\n    e = _enfermedad(s, \\\"tomate\\\")\\n    if re.search(r\\\"miner|insect|uncategor|other|class 16\\\", s):\\n        return \\\"tomate\\\"   # plagas sin clase propia: solo cuentan como tomate para el router\\n    return f\\\"tomate_{e}\\\" if e else \\\"tomate\\\"\\n\\n\\ndef papa_indonesia(carpeta, cfg):\\n    s = _norm(carpeta)\\n    if \\\"healthy\\\" in s:\\n        return \\\"papa_sana\\\"\\n    if \\\"phytophthora\\\" in s:\\n        return \\\"papa_tizon_tardio\\\"\\n    return \\\"papa\\\"   # bacteria, fungi, nematode, pest, virus: sin equivalente exacto\\n\\n\\ndef solo_tomate(carpeta, cfg):\\n    return \\\"tomate\\\"\\n\\n\\ndef cafe(carpeta, cfg):\\n    s = _norm(carpeta)\\n    if re.search(r\\\"healthy|sana|sano|saludable\\\", s):\\n        return \\\"cafe_sana\\\"\\n    if re.search(r\\\"rust|roya|hemileia\\\", s):\\n        return \\\"cafe_roya\\\"\\n    if re.search(r\\\"mycena|citricolor|ojo|spot|mancha\\\", s):\\n        return \\\"cafe_ojo_de_gallo\\\"\\n    return \\\"cafe\\\"\\n\\n\\ndef cacao_monilia(carpeta, cfg):\\n    # carpetas h0 (sana) y m1, m2, m3 (estadios de Moniliophthora roreri)\\n    s = _norm(carpeta)\\n    if re.search(r\\\"\\\\bh0\\\\b|healthy|sana\\\", s):\\n        return \\\"cacao_sana\\\"\\n    if re.search(r\\\"\\\\bm[123]\\\\b|monilia\\\", s):\\n        return \\\"cacao_monilia\\\"\\n    return \\\"cacao\\\"\\n\\n\\ndef cacao_yolo(nombre_clase, cfg):\\n    s = _norm(nombre_clase)\\n    if re.search(r\\\"healthy|sana|sano\\\", s):\\n        return \\\"cacao_sana\\\"\\n    if \\\"monilia\\\" in s:\\n        return \\\"cacao_monilia\\\"\\n    if re.search(r\\\"phytophthora|black|negra\\\", s):\\n        return \\\"cacao_mazorca_negra\\\"\\n    return \\\"cacao\\\"\\n\\n\\ndef no_planta(carpeta, cfg):\\n    s = _norm(carpeta)\\n    if re.search(r\\\"flower|fruit\\\", s):\\n        return \\\"otra_planta\\\"\\n    if re.search(r\\\"airplane|car|cat|dog|motorbike|person\\\", s):\\n        return \\\"no_planta\\\"\\n    return None\\n\\n\\ndef etiqueta_para_modelo(etiqueta, modelo, clases):\\n    \\\"\\\"\\\"Traduce una etiqueta general a la clase de un modelo concreto (o None).\\\"\\\"\\\"\\n    if modelo == \\\"router\\\":\\n        if etiqueta in clases:\\n            return etiqueta\\n        pref = etiqueta.split(\\\"_\\\")[0]\\n        return pref if pref in CULTIVOS else None\\n    return etiqueta if etiqueta in clases else None\\n\"}")
for nombre, texto in ARCHIVOS.items():
    open(os.path.join('fitolens_ent', nombre), 'w', encoding='utf-8').write(texto)
print(sorted(ARCHIVOS))

In [ ]:
!python -m fitolens_ent.entrenar --modelo router

In [ ]:
import json, glob
for f in sorted(glob.glob('/kaggle/working/salida/*/metricas.json')):
    m = json.load(open(f))
    c = m['campo']
    print(f"{m['modelo']:11} {m['backbone']:32} campo: acc={c['accuracy']} IC95={c['accuracy_ic95']} "
          f"f1={c['f1_macro']} n={c['n']} | umbral {c['con_umbral']} | {m['latencia_ms']} ms")
!rm -rf /kaggle/working/fitolens/fuentes /kaggle/working/fitolens/cache  # deja solo los resultados en Output